In [0]:
%run /Workspace/Users/leo346662@gmail.com/Configuration_notebook

In [0]:
%run /Workspace/Users/leo346662@gmail.com/helper

In [0]:
dbutils.widgets.text("pipeline_name", "")
dbutils.widgets.text("notebook_name", "")
dbutils.widgets.text("run_id", "")
dbutils.widgets.text("source", "")
dbutils.widgets.text("input_date", "")


pipeline_name = dbutils.widgets.get("pipeline_name")
notebook_name = dbutils.widgets.get("notebook_name")
run_id = dbutils.widgets.get("run_id")
source = dbutils.widgets.get("source")
input_date = dbutils.widgets.get("input_date")

In [0]:
parsed_date = datetime.strptime(input_date, "%Y%m%d")
prev_month_date = parsed_date - relativedelta(months=1)
prev_month_full_name = prev_month_date.strftime("%B")
print(parsed_date)
print(prev_month_date)
print(prev_month_full_name)

2026-09-29 00:00:00
2026-08-29 00:00:00
August


Total Revenue by Category

In [0]:
spark.sql("DELETE FROM gold.agg_category_sales")

DataFrame[num_affected_rows: bigint]

In [0]:
spark.sql("""
INSERT INTO gold.agg_category_sales
SELECT 
    p.category_code,
    p.category,
    SUM(s.revenue) AS total_revenue,
    SUM(s.quantity) AS total_quantity,
    'August' AS load_month,
    current_timestamp() AS created_at
FROM silver.dim_sales s
LEFT JOIN silver.dim_product p 
       ON s.product_code = p.product_code AND s.category_code = p.category_code
GROUP BY p.category_code, p.category
""")

DataFrame[num_affected_rows: bigint, num_inserted_rows: bigint]

In [0]:
%sql
select * from gold.agg_category_sales

category_code,category,total_revenue,total_quantity,load_month,created_at
A2,1.5T,7200000.0,288,August,2026-09-29T08:21:29.993585Z
WM3,Normal,7050000.0,282,August,2026-09-29T08:21:29.993585Z
FR3,Double,7575000.0,303,August,2026-09-29T08:21:29.993585Z
T3,55IN,7500000.0,300,August,2026-09-29T08:21:29.993585Z
A1,1T,1.014E7,338,August,2026-09-29T08:21:29.993585Z
T1,32IN,5560000.0,278,August,2026-09-29T08:21:29.993585Z
T2,43IN,1.1515E7,329,August,2026-09-29T08:21:29.993585Z
FR1,Small,7475000.0,299,August,2026-09-29T08:21:29.993585Z
T4,85IN,3.888E7,324,August,2026-09-29T08:21:29.993585Z
A3,2T,8125000.0,325,August,2026-09-29T08:21:29.993585Z


Databricks visualization. Run in Databricks to view.

##day wise sales

In [0]:
spark.sql("""
INSERT INTO gold.agg_branch_revenue
SELECT 
    b.branch_id,
    b.branch_area,
    SUM(s.revenue) AS total_revenue,
    SUM(s.quantity) AS total_quantity,
    'August' AS load_month,
    current_timestamp() AS created_at
FROM silver.dim_sales s
LEFT JOIN silver.dim_branch b 
       ON s.branch_id = b.branch_id
GROUP BY b.branch_id, b.branch_area
""")

DataFrame[num_affected_rows: bigint, num_inserted_rows: bigint]

In [0]:
%sql
select * from gold.agg_branch_revenue

branch_id,branch_area,total_revenue,total_quantity,load_month,created_at
CHN_B3,ECR,1.718E7,505,August,2026-09-29T08:43:59.824621Z
BLR_B1,HSR,1.8935E7,499,August,2026-09-29T08:43:59.824621Z
CHN_B4,ADYAR,1.8115E7,507,August,2026-09-29T08:43:59.824621Z
CHN_B2,OMR,1.7215E7,497,August,2026-09-29T08:43:59.824621Z
BLR_B3,WHITEFIELD,1.9125E7,532,August,2026-09-29T08:43:59.824621Z
CHN_B1,CENTRAL,1.5515E7,467,August,2026-09-29T08:43:59.824621Z
BLR_B4,EC_CITY,1.73E7,504,August,2026-09-29T08:43:59.824621Z
BLR_B2,MAJESTIC,1.754E7,508,August,2026-09-29T08:43:59.824621Z


Databricks visualization. Run in Databricks to view.

In [0]:
spark.sql("""
INSERT INTO gold.agg_daily_sales_by_dayname
SELECT  
    invoice_dt AS sales_date,
    date_format(invoice_dt,'EEEE') AS day_name,
    SUM(s.revenue) AS total_revenue,
    SUM(s.quantity) AS total_quantity,
    'August' AS load_month,
    current_timestamp() AS created_at
FROM silver.dim_sales s
GROUP BY invoice_dt, date_format(invoice_dt,'EEEE')
""")

DataFrame[num_affected_rows: bigint, num_inserted_rows: bigint]

In [0]:
%sql
select * from  gold.agg_daily_sales_by_dayname

sales_date,day_name,total_revenue,total_quantity,load_month,created_at
2025-09-03,Wednesday,5650000.0,150,August,2026-09-30T09:03:00.492067Z
2025-09-10,Wednesday,4285000.0,128,August,2026-09-30T09:03:00.492067Z
2025-09-17,Wednesday,4735000.0,145,August,2026-09-30T09:03:00.492067Z
2025-09-24,Wednesday,4970000.0,149,August,2026-09-30T09:03:00.492067Z
2025-09-04,Thursday,4800000.0,129,August,2026-09-30T09:03:00.492067Z
2025-09-06,Saturday,4175000.0,120,August,2026-09-30T09:03:00.492067Z
2025-09-11,Thursday,5180000.0,140,August,2026-09-30T09:03:00.492067Z
2025-09-13,Saturday,5600000.0,152,August,2026-09-30T09:03:00.492067Z
2025-09-18,Thursday,3670000.0,115,August,2026-09-30T09:03:00.492067Z
2025-09-20,Saturday,5305000.0,143,August,2026-09-30T09:03:00.492067Z


Databricks visualization. Run in Databricks to view.

most preferred TVs by revenue and quantity

In [0]:
spark.sql("DELETE FROM gold.agg_tv_product_sales")

DataFrame[num_affected_rows: bigint]

In [0]:
%sql
INSERT INTO gold.agg_tv_product_sales
SELECT 
    p.category, 
    p.product_code,
    SUM(s.quantity) AS total_quantity,
    SUM(s.revenue) AS total_revenue,
    'August' AS load_month,
    current_timestamp() AS created_at
FROM silver.dim_sales s
LEFT JOIN silver.dim_product p 
       ON s.product_code = p.product_code AND s.Category_Code = p.category_code
WHERE p.product_code = 'TV'       -- filter only TV category
GROUP BY p.category, p.product_code;

num_affected_rows,num_inserted_rows
4,4


In [0]:
%sql
select * from gold.agg_tv_product_sales

category,product_code,total_quantity,total_revenue,load_month,created_at
85IN,TV,324,3.888E7,August,2026-09-30T09:14:21.292943Z
55IN,TV,300,7500000.0,August,2026-09-30T09:14:21.292943Z
32IN,TV,278,5560000.0,August,2026-09-30T09:14:21.292943Z
43IN,TV,329,1.1515E7,August,2026-09-30T09:14:21.292943Z


Databricks visualization. Run in Databricks to view.

which product is going good in branch

In [0]:
%sql
delete from gold.agg_branch_product_revenue

num_affected_rows
0


In [0]:
%sql
INSERT INTO gold.agg_branch_product_revenue
SELECT 
      s.branch_id,
      b.branch_area,
      s.product_code,
      p.category,
      SUM(s.revenue) AS total_revenue,
      SUM(s.quantity) AS total_quantity
  FROM silver.dim_sales s
  LEFT JOIN silver.dim_branch b 
         ON s.branch_id = b.branch_id
  LEFT JOIN silver.dim_product p 
         ON s.product_code = p.product_code AND s.category_code = p.category_code
  GROUP BY s.branch_id, b.branch_area, s.product_code, p.category

num_affected_rows,num_inserted_rows
104,104


In [0]:
%sql
select * from gold.agg_branch_product_revenue

branch_id,branch_area,product_code,category,total_revenue,total_quantity
BLR_B2,MAJESTIC,TV,85IN,4680000.0,39
BLR_B3,WHITEFIELD,Fridge,Double,1100000.0,44
BLR_B3,WHITEFIELD,TV,85IN,5640000.0,47
CHN_B3,ECR,Washing-Machine,Automatic,1050000.0,42
BLR_B2,MAJESTIC,AC,1.5T,975000.0,39
CHN_B2,OMR,Fridge,Big,1125000.0,25
BLR_B3,WHITEFIELD,AC,1.5T,925000.0,37
BLR_B2,MAJESTIC,TV,43IN,1435000.0,41
CHN_B1,CENTRAL,AC,2T,1050000.0,42
BLR_B4,EC_CITY,AC,1.5T,900000.0,36


Databricks visualization. Run in Databricks to view.

In [0]:
success='All reports has been generated successfully'
dbutils.notebook.exit(success)